# Climate Risk & Health Prediction

**Goal:** predict `is_climate_sensitive` for each death record (Uganda).
**Metric:** `0.60 × F1 + 0.40 × ROC-AUC`, with the binary label taken at the **default threshold 0.5**.

**Competition rules this notebook follows** (from the Zindi page of the sister AI4EAC edition — confirm them on your own competition's rules page):

- No probability threshold may be set: `TargetF1` = `TargetRAUC >= 0.5`, and `TargetRAUC` must be the **raw** model probability (no post-hoc shifting or rounding).
- Only the datasets provided by the challenge are used (no external data).
- Seeds are fixed so that re-running gives the same submission.
- Two submissions must be selected before the end; the private leaderboard (70–80 % of the test set) decides, so selection is based on cross-validation, not on the small public leaderboard.

**What this notebook does**

1. Loads the merged train / test tables (builds them from the raw files if missing).
2. Builds the final feature set (age flags and age × year interactions on top of the base variables).
3. Evaluates Extra Trees with **location-grouped cross-validation, repeated over 3 seeds**, scoring *raw* probabilities at threshold 0.5.
4. Picks the positive-class training weight (a model hyper-parameter) by cross-validation.
5. Writes `submission_v4.csv` and runs sanity checks.
6. Shows where the remaining errors are, and summarises what was tried without gain.

In [1]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

# Locate the project root (the folder that contains src/config.py)
ROOT = Path.cwd()
while not (ROOT / "src" / "config.py").exists():
    if ROOT.parent == ROOT:
        raise FileNotFoundError("Run this notebook from inside the project (src/config.py not found).")
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.config import (DATA_PROCESSED, DATA_RAW, ID_COL, LOCATION_COL, RANDOM_STATE,
                        TARGET, TEST_MERGED, TRAIN_MERGED)
from src.features import prepare_features
from src.metrics import competition_score
from src.models import make_builder
from src.submission import make_submission
from src.validation import cross_validate, make_folds

np.random.seed(RANDOM_STATE)
print("Project root:", ROOT)

Project root: /home/ferdinand/Documents/Zindi/Projet2/climate-risk-health-prediction


## 1. Data

In [2]:
# Merge the climate features onto Train / Test by ID if the merged files are missing
if not (TRAIN_MERGED.exists() and TEST_MERGED.exists()):
    DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
    climate = pd.read_csv(DATA_RAW / "climate_features.csv")
    for raw_name, out_path in (("Train.csv", TRAIN_MERGED), ("Test.csv", TEST_MERGED)):
        (pd.read_csv(DATA_RAW / raw_name)
           .merge(climate, on=ID_COL, how="left", suffixes=("", "_climate"))
           .to_csv(out_path, index=False))

train = pd.read_csv(TRAIN_MERGED)
test = pd.read_csv(TEST_MERGED)
y = train[TARGET]

print("train:", train.shape, "| test:", test.shape)
print("positive rate (train): %.3f" % y.mean())
print("locations: train %d, test %d, test rows in a location seen in train: %.1f %%"
      % (train[LOCATION_COL].nunique(), test[LOCATION_COL].nunique(),
         100 * test[LOCATION_COL].isin(train[LOCATION_COL]).mean()))

train: (3146, 30) | test: (1030, 29)
positive rate (train): 0.651
locations: train 39, test 11, test rows in a location seen in train: 2.5 %


## 2. Features

Base variables (demographics, calendar, climate windows) plus features that encode what the data analysis showed:

- **Age is the dominant signal** (under-5s are ~84 % positive, over-70s ~29 %), so age flags and `log(age)` are added.
- **The age × period interaction is strong:** for infants under 1 year the share of positives falls from ~97 % (2009–2016) to ~42–59 % (2018–2021). Explicit interaction terms (`age × year`, `infant_post2017`) make this easy for trees to use.
- `hot_days_30d` is constant (0 on every row), so it is dropped.
- Location variables (name, latitude, longitude, elevation, slope) are dropped: almost no test location is seen in train, and the residual location effect is indistinguishable from noise.

In [3]:
def make_features(df):
    X = prepare_features(df, feature_set="base", drop_location_features=True)
    X = X.drop(columns=["hot_days_30d"], errors="ignore")

    age = X["age"].astype(float)
    year = X["death_year"].astype(float)

    X["age_log"] = np.log1p(age)
    X["age_bucket"] = np.digitize(age, [1, 5, 15, 50, 70])
    X["is_u5"] = (age < 5).astype(int)
    X["is_infant"] = (age < 1).astype(int)
    X["is_old"] = (age >= 60).astype(int)

    X["infant_post2017"] = ((age < 1) & (year >= 2018)).astype(int)
    X["age_x_year"] = age * year
    X["infant_x_year"] = X["is_infant"] * year
    X["u5_x_year"] = X["is_u5"] * year
    X["old_x_year"] = X["is_old"] * year
    return X


# Features never use the target, so train and test are built together
all_rows = pd.concat([train.drop(columns=[TARGET]), test], ignore_index=True)
X_all = make_features(all_rows)
X_train = X_all.iloc[: len(train)].reset_index(drop=True)
X_test = X_all.iloc[len(train):].reset_index(drop=True)

print("n features:", X_train.shape[1])
print(list(X_train.columns))

n features: 33
['zone', 'gender', 'age', 'avg_temperature', 'max_temperature', 'min_temperature', 'precipitation', 'max_daily_rain_30d', 'ndvi_30d', 'ndvi_90d', 'rain_days_30d', 'rain_sum_30d', 'rain_sum_7d', 'rain_sum_90d', 'tavg_30d', 'tavg_7d', 'tavg_90d', 'temp_range_mean_30d', 'tmax_30d', 'tmin_30d', 'death_year', 'death_month', 'death_dayofyear', 'age_log', 'age_bucket', 'is_u5', 'is_infant', 'is_old', 'infant_post2017', 'age_x_year', 'infant_x_year', 'u5_x_year', 'old_x_year']


## 3. Validation protocol

- **5-fold `StratifiedGroupKFold` grouped by location**: a location never appears in both training and validation, which mimics the test set (almost every test location is new).
- Repeated with **3 different fold seeds** (42, 7, 2024) because the fold-to-fold standard deviation of the AUC (~0.01) is larger than most differences between candidates.
- Scores use the **raw probabilities at threshold 0.5**, exactly like the competition.

In [4]:
SEEDS = [42, 7, 2024]


def score_raw(y_true, prob):
    """AUC, F1 at the fixed 0.5 threshold, and the weighted competition score."""
    c = competition_score(y_true, prob, threshold=0.5)
    return c["ROC_AUC"], c["F1"], c["Competition_Score"]

## 4. Model and class weight

**Model:** Extra Trees, strongly regularised (`min_samples_leaf=30`, `max_features=0.5`) — the best algorithm in the comparison (CatBoost, XGBoost, LightGBM, Random Forest and blends were not better).

**Class weight:** the positive class is the majority (65 %) and F1 rewards recall, so the best decision point is below 0.5. Rather than moving the threshold or shifting the probabilities afterwards (not allowed), the positive class is **up-weighted during training**. This is an ordinary model hyper-parameter chosen by cross-validation; the probabilities submitted are the model's raw output. The AUC is unchanged by the weight.

In [5]:
WEIGHTS = [1.0, 1.5, 2.0, 2.5, 3.0]
BASE_PARAMS = {"n_estimators": 800, "min_samples_leaf": 30, "max_features": 0.5}

rows, store = [], {}
for w in WEIGHTS:
    params = {**BASE_PARAMS, "class_weight": {0: 1.0, 1: w}}
    seed_scores, oofs, tests = [], [], []
    for seed in SEEDS:
        folds = make_folds(y, "group", groups=train[LOCATION_COL], seed=seed)
        res = cross_validate(make_builder("extra_trees", X_train, params, seed=RANDOM_STATE),
                             X_train, y, folds, X_test)
        seed_scores.append(score_raw(y, res["oof"]))
        oofs.append(res["oof"])
        tests.append(res["test_pred"])

    s = np.array(seed_scores)
    oof_mean, test_mean = np.mean(oofs, axis=0), np.mean(tests, axis=0)
    store[w] = {"oof": oof_mean, "test": test_mean}
    rows.append({"positive_class_weight": w,
                 "AUC": s[:, 0].mean(), "F1_at_0.5": s[:, 1].mean(),
                 "score": s[:, 2].mean(), "score_std_over_seeds": s[:, 2].std(),
                 "test_share_predicted_1": (test_mean >= 0.5).mean()})
    print(f"weight {w:.1f}: score {s[:, 2].mean():.4f}", flush=True)

results = pd.DataFrame(rows).round(4)
best_w = float(results.loc[results["score"].idxmax(), "positive_class_weight"])
print("\nSelected positive-class weight:", best_w)
results

weight 1.0: score 0.8158
weight 1.5: score 0.8210
weight 2.0: score 0.8229
weight 2.5: score 0.8191
weight 3.0: score 0.8151

Selected positive-class weight: 2.0


,positive_class_weight,AUC,F1_at_0.5,score,score_std_over_seeds,test_share_predicted_1
0,1.0,0.8264,0.8087,0.8158,0.0012,0.6699
1,1.5,0.8264,0.8174,0.8210,0.0004,0.8117
2,2.0,0.8264,0.8205,0.8229,0.0001,0.8777
3,2.5,0.8264,0.8142,0.8191,0.0005,0.9107
4,3.0,0.8265,0.8075,0.8151,0.0010,0.9427


## 5. Final submission

Test probabilities are the average of the 15 fold models (5 folds × 3 seeds). The label is `probability >= 0.5` — nothing else is applied.

In [6]:
final_prob = store[best_w]["test"]
submission, path = make_submission(test[ID_COL].values, final_prob, threshold=0.5,
                                   filename="submission_v4.csv")
print("saved to:", path)
submission.head()

saved to: /home/ferdinand/Documents/Zindi/Projet2/climate-risk-health-prediction/outputs/submissions/submission_v4.csv


,ID,TargetF1,TargetRAUC
0,ID_E760D84B,1,0.569254
1,ID_6EDEA907,1,0.756610
2,ID_B9FFC8D8,1,0.583569
3,ID_74C6C94E,1,0.540459
4,ID_0E02825D,1,0.785179


In [7]:
# Sanity checks against the sample submission and the competition rules
sample = pd.read_csv(DATA_RAW / "SampleSubmission.csv")

assert list(submission.columns) == list(sample.columns), "column names differ from the sample"
assert submission.shape == sample.shape, "row count differs from the sample"
assert (submission[ID_COL].values == sample[ID_COL].values).all(), "ID order differs from the sample"
assert ((submission["TargetRAUC"] >= 0.5).astype(int) == submission["TargetF1"]).all(), \
    "labels must be the raw probabilities at threshold 0.5"
assert submission["TargetRAUC"].between(0, 1).all() and not submission.isna().any().any()

print("All checks passed.")
print("share of label 1 in test: %.3f (train positive rate %.3f)"
      % (submission["TargetF1"].mean(), y.mean()))

All checks passed.
share of label 1 in test: 0.878 (train positive rate 0.651)


## 6. Where the remaining error is

Three segments hold about 80 % of the errors, and inside each of them the model barely separates the classes. Climate knowledge is most valuable here: what kinds of death are climate-sensitive for adults, and what changed in 2018 for infants?

In [8]:
from sklearn.metrics import roc_auc_score

oof_best = store[best_w]["oof"]


def segment(df):
    age = df["age"].to_numpy()
    year = pd.to_datetime(df["deathdate"]).dt.year.to_numpy()
    age_band = np.where(age < 1, "<1", np.where(age < 5, "1-4", np.where(age < 50, "5-49", "50+")))
    era = np.where(year <= 2016, "2007-16", np.where(year == 2017, "2017", "2018+"))
    return pd.Series(age_band + " | " + era)


seg_train, seg_test = segment(train), segment(test)
frame = pd.DataFrame({"segment": seg_train, "y": y.to_numpy(), "p": oof_best})

rows = []
for name, g in frame.groupby("segment"):
    auc = roc_auc_score(g["y"], g["p"]) if g["y"].nunique() > 1 else np.nan
    errors = int(((g["p"] >= 0.5).astype(int) != g["y"]).sum())
    rows.append({"segment": name, "n_train": len(g), "positive_rate": g["y"].mean(),
                 "AUC_inside_segment": auc, "errors": errors,
                 "% of test": 100 * (seg_test == name).mean()})

(pd.DataFrame(rows).sort_values("errors", ascending=False).round(3).reset_index(drop=True))

,segment,n_train,positive_rate,AUC_inside_segment,errors,% of test
0,50+ | 2007-16,655,0.331,0.635,246,20.000
1,<1 | 2018+,492,0.563,0.553,215,14.369
2,5-49 | 2007-16,506,0.575,0.591,214,18.350
3,5-49 | 2018+,111,0.432,0.669,62,3.495
4,1-4 | 2007-16,556,0.926,0.542,41,17.573
5,50+ | 2018+,97,0.278,0.702,21,3.107
6,<1 | 2007-16,598,0.970,0.419,18,19.126
7,5-49 | 2017,25,0.360,0.674,16,0.291
8,1-4 | 2018+,56,0.875,0.621,7,2.136
9,50+ | 2017,19,0.316,0.667,4,0.291


## 7. What was tried (cross-validation, location-grouped)

Scores below are `0.6 × F1 + 0.4 × AUC` at the **best threshold on out-of-fold predictions** (used only for comparing candidates; the submission itself uses the raw probabilities at 0.5). Differences below ~0.003 are within noise.

| Experiment | Score | Verdict |
|---|---|---|
| Extra Trees, original features | 0.8197 | reference |
| + age flags and age × year features, leaf 30 | 0.8232 | **kept** (consistent over 3 seeds) |
| Stronger regularisation (leaf 10 → 30) | +0.002 | kept |
| XGBoost / LightGBM / CatBoost / Random Forest | 0.8206 / 0.8197 / 0.8167 / 0.8214 | not better |
| Blends (ET + XGB, + LGB, + CatBoost) | 0.8232 / 0.8227 / 0.8218 | no gain |
| No weather variables at all | 0.8221 | weather adds ≈ 0.001 |
| Same-day weather only / 30-90 day windows only | 0.8219 / 0.8233 | no difference |
| Separate models per age group | 0.8216 | no gain |
| Random search on Extra Trees (10 configs) | ≤ 0.8227 | no gain |
| Temporal neighbour target encoding | 0.8192 | no gain |
| Seasonal (sin/cos), non-overlapping windows, anomalies, age × climate | ±0.002 | no gain |
| Location effect (residual variance vs permutation) | — | indistinguishable from noise |
| Synthetic data (SMOTE-NC inside the folds) | 0.8209 (AUC 0.8227) | **worse**; doubling with SMOTE: 0.8194 |
| Post-hoc probability shift to make 0.5 optimal | 0.8235 | **not used** — equivalent to setting a threshold |

## 8. Next steps

- Check the rules page of your own competition (threshold, external data, eligibility) before submitting.
- Select the two final submissions using cross-validation; the public leaderboard covers only ~20–30 % of the test rows and is noisy.
- Domain work with the highest potential: the three segments in section 6, especially the change in infant labels from 2018 on.